# Face Analysis System (Age, Gender, Emotion)

This notebook is a complete Computer Vision project for **face analysis** with three outputs:
- Age group (`0-9 ... 70+`)
- Gender (`male/female`)
- Emotion (`angry, disgust, fear, happy, sad, surprise, neutral`)

## 0. Project Overview

### Design decisions
- **Only one pretrained component**: MediaPipe Face Detector.
- Task models are trained **from scratch** in TensorFlow/Keras.
- Two modes are supported:
  - `train`: train and save new models.
  - `quick-infer`: skip training and load saved `.keras` checkpoints.

## Expected outcomes
- End-to-end training and evaluation in Colab.
- Image and video inference demos with overlays.
- Clear markdown documentation and reproducibility notes.

## 1. Environment Setup

This section installs dependencies, sets random seeds, and checks runtime hardware.

In [ ]:
# Core dependencies for Colab
!pip -q install kaggle mediapipe opencv-python-headless seaborn scikit-learn

import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('OpenCV:', cv2.__version__)
print('NumPy:', np.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Choose mode: 'train' or 'quick-infer'
RUN_MODE = 'train'  # change to 'quick-infer' after checkpoints are available

PROJECT_ROOT = Path('/content')
DATA_ROOT = PROJECT_ROOT / 'data'
SRC_ROOT = PROJECT_ROOT / 'src'
ARTIFACT_ROOT = Path('/content/drive/MyDrive/face_analysis_artifacts')
CHECKPOINT_ROOT = ARTIFACT_ROOT / 'checkpoints'

print('RUN_MODE:', RUN_MODE)

## 2. Kaggle + Google Drive Setup

- Drive is used for persistent checkpoints and logs.
- Kaggle API is used to download datasets (`UTKFace`, `FER2013`).

In [ ]:
from google.colab import drive, files

# Persistent storage for checkpoints and logs
drive.mount('/content/drive')
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

# Kaggle token setup
kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    print('Upload your kaggle.json file')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('kaggle.json was not uploaded')
    kaggle_json.write_bytes(uploaded['kaggle.json'])

os.chmod(kaggle_json, 0o600)
print('Kaggle token ready:', kaggle_json.exists())

## 3. Dataset Download (Kaggle API)

Fallback slug lists are used in case a dataset owner/slug changes.

In [ ]:
import shutil
import subprocess

DATA_ROOT.mkdir(parents=True, exist_ok=True)


def try_download(slug_candidates, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    last_error = None
    for slug in slug_candidates:
        cmd = ['kaggle', 'datasets', 'download', '-d', slug, '-p', str(out_dir), '--unzip']
        print('Trying:', slug)
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode == 0:
            print('Downloaded:', slug)
            return slug
        last_error = res.stderr.strip()
    raise RuntimeError(f'Failed all slugs {slug_candidates}. Last error: {last_error}')


def find_utkface_dir(search_root: Path) -> Path:
    # Find a directory with many files that match UTKFace naming pattern.
    image_files = list(search_root.rglob('*.jpg')) + list(search_root.rglob('*.jpeg')) + list(search_root.rglob('*.png'))
    if not image_files:
        raise FileNotFoundError('No image files found for UTKFace.')

    import re
    pattern = re.compile(r'^(\d{1,3})_(\d)_(\d)_.+')

    counts = {}
    for img in image_files:
        if pattern.match(img.name):
            counts[img.parent] = counts.get(img.parent, 0) + 1

    if not counts:
        raise FileNotFoundError('No UTKFace-style filenames found.')

    best_dir = max(counts, key=counts.get)
    print('UTKFace directory selected:', best_dir, 'matched files:', counts[best_dir])
    return best_dir


def find_fer_csv(search_root: Path) -> Path:
    candidates = list(search_root.rglob('fer2013.csv'))
    if not candidates:
        raise FileNotFoundError('fer2013.csv not found after download')
    return candidates[0]


utk_download_dir = DATA_ROOT / 'utkface_raw'
fer_download_dir = DATA_ROOT / 'fer2013_raw'

utk_slug = try_download(
    ['jangedoo/utkface-new', 'kaggledev/utkface', 'datasetsandata/utkface-new'],
    utk_download_dir,
)
fer_slug = try_download(
    ['msambare/fer2013', 'deadskull7/fer2013'],
    fer_download_dir,
)

UTKFACE_ROOT = find_utkface_dir(utk_download_dir)
FER_CSV_PATH = find_fer_csv(fer_download_dir)

print('UTK slug:', utk_slug)
print('FER slug:', fer_slug)
print('UTKFACE_ROOT:', UTKFACE_ROOT)
print('FER_CSV_PATH:', FER_CSV_PATH)

## 4. Data Cleaning and Label Engineering

We use `src/data_pipeline.py` to:
- parse UTKFace file names,
- map ages into 8 bins,
- validate images,
- split FER2013 into train/val/test,
- build clean DataFrames before tf.data conversion.

In [ ]:
# If your src/ folder is not present in Colab, upload the project files first.
if not Path('src').exists():
    raise FileNotFoundError('src/ folder not found in current working directory.')

from src.config import AGE_BINS, AGE_GROUP_LABELS, EMOTION_LABELS, GENDER_LABELS, TrainConfig
from src.data_pipeline import (
    build_age_gender_datasets,
    build_emotion_datasets,
    load_fer2013_dataframe,
    load_utkface_dataframe,
)

utk_df = load_utkface_dataframe(str(UTKFACE_ROOT), age_bins=AGE_BINS)
fer_train_df, fer_val_df, fer_test_df = load_fer2013_dataframe(str(FER_CSV_PATH))

print('UTKFace samples:', len(utk_df))
print('FER train/val/test:', len(fer_train_df), len(fer_val_df), len(fer_test_df))
utk_df.head()

## 5. Exploratory Data Analysis (EDA)

Here we inspect class distributions and sample faces/emotions.

In [ ]:
from src.visualize import plot_label_distribution, show_image_grid

plot_label_distribution(utk_df, 'age_group_label', 'UTKFace Age Group Distribution')
plot_label_distribution(utk_df, 'gender_label', 'UTKFace Gender Distribution')
plot_label_distribution(fer_train_df, 'emotion', 'FER2013 Train Emotion Distribution')

In [ ]:
# Show random UTKFace examples with labels
sample_rows = utk_df.sample(n=min(12, len(utk_df)), random_state=SEED)
images = []
titles = []
for _, row in sample_rows.iterrows():
    img = cv2.imread(row['path'])
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (96, 96))
    images.append(img)
    titles.append(f"{row['age_group_label']} | {row['gender_label']}")

show_image_grid(np.array(images), titles=titles, ncols=4, figsize=(12, 8))

In [ ]:
# Show random FER examples
def fer_pixels_to_img(pixel_str: str):
    arr = np.fromstring(pixel_str, sep=' ', dtype=np.uint8)
    if arr.size != 48 * 48:
        return None
    return arr.reshape(48, 48)

fer_sample = fer_train_df.sample(n=min(12, len(fer_train_df)), random_state=SEED)
emo_images = []
emo_titles = []
for _, row in fer_sample.iterrows():
    img = fer_pixels_to_img(row['pixels'])
    if img is None:
        continue
    emo_images.append(img)
    emo_titles.append(EMOTION_LABELS[int(row['emotion'])])

show_image_grid(np.array(emo_images), titles=emo_titles, ncols=4, figsize=(10, 8))

## 6. Preprocessing Pipelines (`tf.data`)

- Age/Gender input: `128x128 RGB`
- Emotion input: `64x64 grayscale`
- Augmentations: horizontal flip, brightness/contrast jitter

In [ ]:
cfg = TrainConfig()

train_ag_ds, val_ag_ds, test_ag_ds = build_age_gender_datasets(utk_df, batch_size=cfg.batch_size)
train_emo_ds, val_emo_ds, test_emo_ds = build_emotion_datasets(
    train_df=fer_train_df,
    val_df=fer_val_df,
    test_df=fer_test_df,
    batch_size=cfg.batch_size,
)

print('Age/Gender dataset objects ready')
print('Emotion dataset objects ready')

# Smoke checks for tensor shapes
x_ag, y_ag = next(iter(train_ag_ds))
print('AG batch image shape:', x_ag.shape)
print('AG age labels shape:', y_ag['age_output'].shape)
print('AG gender labels shape:', y_ag['gender_output'].shape)

x_em, y_em = next(iter(train_emo_ds))
print('Emotion batch image shape:', x_em.shape)
print('Emotion labels shape:', y_em.shape)

## 7. Model Definitions

- `build_age_gender_model`: shared CNN backbone + two output heads.
- `build_emotion_model`: CNN classifier for FER emotion labels.

In [ ]:
from src.models import build_age_gender_model, build_emotion_model

age_gender_model = build_age_gender_model(input_shape=(128, 128, 3), num_age_classes=len(AGE_GROUP_LABELS))
emotion_model = build_emotion_model(input_shape=(64, 64, 1), num_emotions=len(EMOTION_LABELS))

age_gender_model.summary()
emotion_model.summary()

## 8. Training

Training defaults:
- Optimizer: Adam (`1e-3`)
- ReduceLROnPlateau
- EarlyStopping (`patience=5`)
- Best checkpoints saved as `.keras` in Google Drive

In [ ]:
from src.train import train_age_gender, train_emotion
from src.visualize import plot_training_history

AG_BEST_PATH = CHECKPOINT_ROOT / 'age_gender' / 'age_gender_best.keras'
EMO_BEST_PATH = CHECKPOINT_ROOT / 'emotion' / 'emotion_best.keras'

if RUN_MODE == 'train':
    history_ag = train_age_gender(age_gender_model, train_ag_ds, val_ag_ds, out_dir=str(CHECKPOINT_ROOT))
    history_emo = train_emotion(emotion_model, train_emo_ds, val_emo_ds, out_dir=str(CHECKPOINT_ROOT))

    plot_training_history(history_ag, title='Age+Gender Model')
    plot_training_history(history_emo, title='Emotion Model')

    age_gender_model = tf.keras.models.load_model(AG_BEST_PATH)
    emotion_model = tf.keras.models.load_model(EMO_BEST_PATH)
else:
    if not AG_BEST_PATH.exists() or not EMO_BEST_PATH.exists():
        raise FileNotFoundError('Checkpoints not found. Run with RUN_MODE="train" first.')
    age_gender_model = tf.keras.models.load_model(AG_BEST_PATH)
    emotion_model = tf.keras.models.load_model(EMO_BEST_PATH)

print('Loaded models:')
print(' -', AG_BEST_PATH)
print(' -', EMO_BEST_PATH)

## 9. Evaluation

This section computes:
- Accuracy metrics
- Classification reports
- Confusion matrices
- Threshold checks vs project targets

In [ ]:
from src.evaluate import evaluate_age_gender, evaluate_emotion
from src.visualize import plot_confusion_matrix

ag_results = evaluate_age_gender(age_gender_model, test_ag_ds, age_group_labels=AGE_GROUP_LABELS)
emo_results = evaluate_emotion(emotion_model, test_emo_ds, emotion_labels=EMOTION_LABELS)

print(f"Gender accuracy: {ag_results['gender_accuracy']:.4f}")
print(f"Age-group accuracy: {ag_results['age_accuracy']:.4f}")
print(f"Emotion accuracy: {emo_results['emotion_accuracy']:.4f}")

plot_confusion_matrix(ag_results['age_confusion_matrix'], AGE_GROUP_LABELS, 'Age Group Confusion Matrix')
plot_confusion_matrix(ag_results['gender_confusion_matrix'], GENDER_LABELS, 'Gender Confusion Matrix')
plot_confusion_matrix(emo_results['emotion_confusion_matrix'], EMOTION_LABELS, 'Emotion Confusion Matrix')

In [ ]:
# Target checks from project plan
TARGETS = {
    'gender_accuracy': 0.90,
    'age_accuracy': 0.55,
    'emotion_accuracy': 0.60,
}

current = {
    'gender_accuracy': ag_results['gender_accuracy'],
    'age_accuracy': ag_results['age_accuracy'],
    'emotion_accuracy': emo_results['emotion_accuracy'],
}

for name, threshold in TARGETS.items():
    value = current[name]
    status = 'PASS' if value >= threshold else 'FAIL'
    print(f'{name}: {value:.4f} (target {threshold:.2f}) -> {status}')

## 10. Inference Pipeline

Inference flow:
1. Detect faces with MediaPipe.
2. Crop each face.
3. Predict age-group + gender + emotion.
4. Draw overlays on image/frame.

In [ ]:
from src.infer import annotate_image, detect_faces_bgr, load_detector, predict_face, run_video_inference

detector = load_detector()
print('MediaPipe detector initialized')

## 11. Demo: Image Upload

Upload an image and run full face analysis.

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No image uploaded')

image_name = next(iter(uploaded.keys()))
image_path = Path(image_name)

frame = cv2.imread(str(image_path))
if frame is None:
    raise ValueError(f'Cannot read image: {image_path}')

boxes = detect_faces_bgr(frame, detector)
if not boxes:
    print('No faces detected.')
    display(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
else:
    preds = []
    for box in boxes:
        crop = frame[box.y1:box.y2, box.x1:box.x2]
        if crop.size == 0:
            continue
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        pred = predict_face(crop_rgb, age_gender_model, emotion_model)
        preds.append((box, pred))

    out = annotate_image(frame, preds)
    display(Image.fromarray(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)))

## 12. Demo: Video Upload + Overlay

Upload a video file (`.mp4` recommended), run frame-by-frame analysis, and get an annotated output video.

In [ ]:
from IPython.display import HTML
from base64 import b64encode

uploaded_video = files.upload()
if not uploaded_video:
    raise RuntimeError('No video uploaded')

video_name = next(iter(uploaded_video.keys()))
input_video = Path(video_name)
output_video = Path('/content/annotated_output.mp4')

result_path = run_video_inference(
    video_path=str(input_video),
    out_path=str(output_video),
    detector=detector,
    ag_model=age_gender_model,
    emo_model=emotion_model,
)

print('Annotated video saved to:', result_path)

mp4 = open(result_path, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f'<video width=700 controls><source src="{data_url}" type="video/mp4"></video>')

## 13. Error Analysis

This section inspects common failures and helps guide model improvements.

In [ ]:
# Emotion misclassifications (test split)
wrong_idx = np.where(emo_results['y_true'] != emo_results['y_pred'])[0]
print('Emotion misclassified samples:', len(wrong_idx))

preview_n = min(12, len(wrong_idx))
if preview_n > 0:
    subset = fer_test_df.iloc[wrong_idx[:preview_n]].copy().reset_index(drop=True)
    imgs = []
    ttl = []
    for i in range(len(subset)):
        arr = np.fromstring(subset.loc[i, 'pixels'], sep=' ', dtype=np.uint8)
        if arr.size != 48 * 48:
            continue
        img = arr.reshape(48, 48)
        imgs.append(img)
        y_t = EMOTION_LABELS[int(emo_results['y_true'][wrong_idx[i]])]
        y_p = EMOTION_LABELS[int(emo_results['y_pred'][wrong_idx[i]])]
        ttl.append(f'T:{y_t} | P:{y_p}')

    if imgs:
        show_image_grid(np.array(imgs), titles=ttl, ncols=4, figsize=(11, 8))

In [ ]:
# Top confusions for age groups
age_cm = ag_results['age_confusion_matrix'].copy()
np.fill_diagonal(age_cm, 0)
flat_idx = np.argsort(age_cm, axis=None)[::-1]
print('Top age-group confusions:')
shown = 0
for idx in flat_idx:
    r, c = np.unravel_index(idx, age_cm.shape)
    if age_cm[r, c] == 0:
        break
    print(f'{AGE_GROUP_LABELS[r]} -> {AGE_GROUP_LABELS[c]}: {age_cm[r, c]}')
    shown += 1
    if shown == 5:
        break

## 14. Reproducibility and Re-run Guide

- Keep `SEED=42` fixed for consistent splits and initialization.
- Save artifacts in Drive under `face_analysis_artifacts`.
- For later sessions set `RUN_MODE='quick-infer'` and load checkpoints.

In [ ]:
import platform

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('NumPy:', np.__version__)
print('Random seed:', SEED)
print('Checkpoint root:', CHECKPOINT_ROOT)
print('Age/Gender checkpoint exists:', AG_BEST_PATH.exists())
print('Emotion checkpoint exists:', EMO_BEST_PATH.exists())

## 15. Ethics, Bias, and Responsible Use

This project is for educational Computer Vision practice.

Important cautions:
- Age and emotion are noisy, context-dependent signals.
- Dataset bias may degrade performance for underrepresented groups.
- Results should **not** be used for high-stakes or rights-impacting decisions.
- Always disclose model uncertainty and limitations when presenting outputs.

### Project Checklist

- [x] End-to-end data pipeline
- [x] Models trained from scratch (except face detector)
- [x] Evaluation with target thresholds
- [x] Image and video demos
- [x] Documented notebook sections